In [3]:
# ==============================================================================
# Cal Regression: gcvEarth Strict + ALL-IN-ONE Format
# - 10-fold CVが全Fold成功したdegreeのみを採択する厳格版
# - 出力形式、メトリクス、Overfit判定を最初のALL-IN-ONE版と統一
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
  library(earth)
  library(pls)
})

closeAllConnections()
set.seed(42)

# -------------------------------
# USER SETTINGS
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
# 28行目付近（target_vars の後あたりに追加）
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'Var.1'
)
today <- format(Sys.Date(), "%Y%m%d")

file_names <- c(
#           　　　　　　　　　　　"n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
#                 "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
#                 "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
                "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv")

# 実行するモデル
models_to_run <- c("gcvEarth")

ood_ratio <- 0.10
overfit_delta_r2_thr   <- 0.10
overfit_delta_rmse_rel <- 0.10 

# -------------------------------
# HELPERS
# -------------------------------
safe_r2   <- function(y, p) { r <- suppressWarnings(cor(y, p)); if (is.na(r)) return(NA_real_); return(as.numeric(r^2)) }
safe_rmse <- function(y, p) Metrics::rmse(y, p)
safe_mae  <- function(y, p) Metrics::mae(y, p)
safe_rpd  <- function(y, rmse_val) { if (is.na(rmse_val) || rmse_val == 0) return(NA_real_); return(as.numeric(sd(y) / rmse_val)) }

clean_features <- function(df, features) {
  if (length(features) == 0) return(NULL)
  v <- sapply(df[, features, drop = FALSE], var, na.rm = TRUE)
  features <- names(v)[v > 1e-10 & !is.na(v)]
  if (length(features) > 1) {
    tmp_cor <- suppressWarnings(cor(df[, features], use = "pairwise.complete.obs"))
    tmp_cor[is.na(tmp_cor)] <- 0
    hc <- findCorrelation(tmp_cor, cutoff = 0.95)
    if (length(hc) > 0) features <- features[-hc]
  }
  return(features)
}

create_ood_split <- function(df, feature_cols, ood_ratio = 0.1) {
  X <- df[, feature_cols, drop = FALSE]
  centroid <- colMeans(X)
  dists <- apply(X, 1, function(row) sqrt(sum((row - centroid)^2)))
  num_ood <- max(1, floor(nrow(df) * ood_ratio))
  ood_indices <- order(dists, decreasing = TRUE)[1:num_ood]
  list(train = df[-ood_indices, , drop = FALSE], test = df[ood_indices, , drop = FALSE])
}

scale_neg1_to_1 <- function(train_df, test_df, feature_cols) {
  pp <- preProcess(train_df[, feature_cols, drop = FALSE], method = "range")
  train_01 <- predict(pp, train_df[, feature_cols, drop = FALSE])
  test_01  <- predict(pp, test_df[, feature_cols, drop = FALSE])
  train_df[, feature_cols] <- train_01 * 2 - 1
  test_df[, feature_cols]  <- test_01 * 2 - 1
  list(train = train_df, test = test_df)
}

extract_oof_best <- function(model, train_scaled) {
  pred_oof <- model$pred
  if (is.null(pred_oof) || nrow(pred_oof) == 0) return(NULL)
  for (pname in names(model$bestTune)) {
    pred_oof <- pred_oof[pred_oof[[pname]] == model$bestTune[[pname]], , drop = FALSE]
  }
  pred_oof$pred <- as.numeric(as.character(pred_oof$pred))
  pred_oof$obs  <- as.numeric(as.character(pred_oof$obs))
  pred_oof_agg <- pred_oof %>% group_by(rowIndex) %>% 
    summarise(Predicted = mean(pred, na.rm=TRUE), Observed = mean(obs, na.rm=TRUE), .groups="drop") %>%
    filter(!is.na(Predicted))
  if (nrow(pred_oof_agg) == 0) return(NULL)
  pred_oof_agg$SampleID <- rownames(train_scaled)[pred_oof_agg$rowIndex]
  pred_oof_agg %>% select(SampleID, Observed, Predicted) %>% mutate(Type = "CV10_OOF")
}

calc_permutation_importance <- function(model, data_with_target, target_col, feature_cols, base_r2) {
  out <- list()
  for (col in feature_cols) {
    temp <- data_with_target; temp[[col]] <- sample(temp[[col]])
    preds <- tryCatch(predict(model, temp[, feature_cols, drop = FALSE]), error = function(e) NA)
    new_r2 <- if (all(is.na(preds))) 0 else safe_r2(temp[[target_col]], preds)
    out[[col]] <- base_r2 - (if (is.na(new_r2)) 0 else new_r2)
  }
  out
}

# -------------------------------
# MAIN RUNNER
# -------------------------------
run_one_model <- function(model_key) {
  model_name <- model_key
  out_dir <- file.path(getwd(), paste0(today, "_Rebuilt_12Files_", model_name, "_Strict_Final"))
  if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

  cat("\n======================================================\n")
  cat(sprintf("--- %s Analysis Started: %s ---\n", model_name, today))
  cat("======================================================\n")

  summary_data <- dplyr::tibble()
  importance_data <- dplyr::tibble()

  for (file in file_names) {
    filepath <- file.path(base_path, file)
    if (!file.exists(filepath)) next
    df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
    if (is.null(df_raw)) next

    cat(sprintf("\n=== Processing: %s ===\n", file))

    for (target in target_vars) {
      if (!target %in% colnames(df_raw)) next
      df_temp <- df_raw[sapply(df_raw, is.numeric)] %>% na.omit()
      if (nrow(df_temp) < 20) next

      # 特徴量抽出（全目的変数を除外）
      features <- setdiff(colnames(df_temp), target_vars)
      # ★追加：m_all系のみ追加の不適切変数を除外
      if (grepl("^m_all", file)) {
        features <- setdiff(features, inappropriate_vars)
        # cat(sprintf("    [Security] Removed inappropriate vars for %s\n", file)) # 確認用
      }
      features <- clean_features(df_temp, features)
      if (is.null(features) || length(features) < 2) next

      # OOD Split & Scaling
      split_res <- create_ood_split(df_temp, features, ood_ratio)
      scaled_res <- scale_neg1_to_1(split_res$train, split_res$test, features)
      train_scaled <- scaled_res$train; test_scaled <- scaled_res$test

      # 分割後再チェック
      features <- clean_features(train_scaled, features)
      if (is.null(features) || length(features) < 2) next

      train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final", allowParallel = FALSE)

      # --- 厳格なDegree選択学習 ---
      set.seed(42)
      model <- NULL; models_ok <- list(); rmse_ok <- c()
      
      for (deg in 1:3) {
        cat(sprintf("  Target=%s | Trying degree=%d\n", target, deg))
        m_tmp <- tryCatch(
          train(x = train_scaled[, features], y = train_scaled[[target]], method = "gcvEarth", 
                trControl = train_ctrl, tuneGrid = expand.grid(degree = deg),
                nk = 80, minspan = 25, penalty = 4, nprune = 40),
          error = function(e) NULL
        )
        if (!is.null(m_tmp) && !is.null(m_tmp$resample) && nrow(m_tmp$resample) == 10) {
          if (!any(is.na(m_tmp$resample$RMSE))) {
            models_ok[[as.character(deg)]] <- m_tmp
            rmse_ok[as.character(deg)] <- m_tmp$results$RMSE[1]
          }
        }
      }
      
      if (length(models_ok) > 0) {
        model <- models_ok[[names(rmse_ok)[which.min(rmse_ok)]]]
        cat(sprintf("    Selected degree=%d as best.\n", model$bestTune$degree))
      } else { next }

      # --- Predictions & OOF ---
      pred_train <- predict(model, train_scaled[, features])
      pred_test  <- predict(model, test_scaled[, features])
      df_oof_out <- extract_oof_best(model, train_scaled)
      if (is.null(df_oof_out)) next

      # --- Metrics Calculation ---
      r2_train  <- safe_r2(train_scaled[[target]], pred_train)
      rmse_train <- safe_rmse(train_scaled[[target]], pred_train)
      mae_train  <- safe_mae(train_scaled[[target]], pred_train)
      rpd_train  <- safe_rpd(train_scaled[[target]], rmse_train)

      r2_cv10   <- safe_r2(df_oof_out$Observed, df_oof_out$Predicted)
      rmse_cv10  <- safe_rmse(df_oof_out$Observed, df_oof_out$Predicted)
      mae_cv10   <- safe_mae(df_oof_out$Observed, df_oof_out$Predicted)
      rpd_cv10   <- safe_rpd(df_oof_out$Observed, rmse_cv10)

      r2_ood    <- safe_r2(test_scaled[[target]], pred_test)
      rmse_ood   <- safe_rmse(test_scaled[[target]], pred_test)
      mae_ood    <- safe_mae(test_scaled[[target]], pred_test)
      rpd_ood    <- safe_rpd(test_scaled[[target]], rmse_ood)

      # --- Overfit Check ---
      delta_r2   <- r2_train - r2_cv10
      delta_rmse <- rmse_cv10 - rmse_train
      overfit_flag <- (is.finite(delta_r2) && delta_r2 >= overfit_delta_r2_thr) ||
                      (is.finite(delta_rmse) && is.finite(rmse_train) && delta_rmse >= overfit_delta_rmse_rel * abs(rmse_train))
      overfit_label <- ifelse(overfit_flag, "Overfit_suspected", "OK")

      # --- Save Outputs ---
      # A. RDS Model
      saveRDS(list(model=model, training_data=train_scaled, features=features, target=target), 
              file.path(out_dir, paste0("fixed_", today, "_", model_name, "_model_", tools::file_path_sans_ext(file), "_", target, ".rds")))

      # B. Predictions CSV
      df_pred_all <- bind_rows(
        data.frame(SampleID=rownames(train_scaled), Observed=train_scaled[[target]], Predicted=as.numeric(pred_train), Type="Train"),
        df_oof_out,
        data.frame(SampleID=rownames(test_scaled), Observed=test_scaled[[target]], Predicted=as.numeric(pred_test), Type="OOD_Test")
      )
      write.csv(df_pred_all, file.path(out_dir, paste0("fixed_", today, "_", model_name, "_", tools::file_path_sans_ext(file), "_", target, "_pred.csv")), row.names=FALSE)

      # C. Importance
      imps <- calc_permutation_importance(model, train_scaled, target, features, r2_train)
      imp_df <- data.frame(File=file, Target=target, Feature=names(imps), Importance_Mean=as.numeric(unlist(imps)), Importance_Type="Permutation_R2drop") %>%
                filter(is.finite(Importance_Mean) & abs(Importance_Mean) > 1e-6)
      importance_data <- bind_rows(importance_data, imp_df)

      # D. Summary Row
      summary_data <- bind_rows(summary_data, data.frame(
        File=file, Target=target, Train_R2=r2_train, Train_RMSE=rmse_train, Train_MAE=mae_train, Train_RPD=rpd_train,
        CV10_R2=r2_cv10, CV10_RMSE=rmse_cv10, CV10_MAE=mae_cv10, CV10_RPD=rpd_cv10,
        OOD_R2=r2_ood, OOD_RMSE=rmse_ood, OOD_MAE=mae_ood, OOD_RPD=rpd_ood,
        n_samples=nrow(df_temp), n_features=length(features), Best_Params=paste0("degree=", model$bestTune$degree, ", nk=80"),
        Overfit_Flag=overfit_flag, Overfit_Label=overfit_label, Delta_R2_TrainMinusCV10=delta_r2, Delta_RMSE_CV10MinusTrain=delta_rmse
      ))
      
      cat(sprintf("    Metrics: Train_R2=%.3f | CV10_R2=%.3f | OOD_R2=%.3f | %s\n", r2_train, r2_cv10, r2_ood, overfit_label))
    }
  }

  # Final CSVs
  write.csv(summary_data, file.path(out_dir, paste0("fixed_", today, "_", model_name, "_summary.csv")), row.names=FALSE)
  write.csv(importance_data, file.path(out_dir, paste0("fixed_", today, "_", model_name, "_feature_importance.csv")), row.names=FALSE)
  
  overfit_tbl <- summary_data %>% select(File, Target, Train_R2, CV10_R2, OOD_R2, Overfit_Flag, Overfit_Label, Delta_R2_TrainMinusCV10, Best_Params)
  write.csv(overfit_tbl, file.path(out_dir, paste0("fixed_", today, "_", model_name, "_overfit_judgement.csv")), row.names=FALSE)
}

for (mk in models_to_run) { run_one_model(mk) }


--- gcvEarth Analysis Started: 20260112 ---

=== Processing: m_all.csv ===


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_Ag, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, Additivesname_PB.C60, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB"


  Target=Jsc | Trying degree=1
  Target=Jsc | Trying degree=2
  Target=Jsc | Trying degree=3
    Selected degree=1 as best.
    Metrics: Train_R2=0.608 | CV10_R2=0.472 | OOD_R2=0.039 | Overfit_suspected


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_Ag, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, Additivesname_PB.C60, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB"


  Target=Voc | Trying degree=1
  Target=Voc | Trying degree=2
  Target=Voc | Trying degree=3
    Selected degree=1 as best.
    Metrics: Train_R2=0.612 | CV10_R2=0.424 | OOD_R2=0.127 | Overfit_suspected


...

  Target=Voc | Trying degree=1
  Target=Voc | Trying degree=2
  Target=Voc | Trying degree=3
    Selected degree=1 as best.
    Metrics: Train_R2=0.696 | CV10_R2=0.514 | OOD_R2=0.629 | Overfit_suspected


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB, X165.2, X165.5"


  Target=FF | Trying degree=1
  Target=FF | Trying degree=2
  Target=FF | Trying degree=3
    Selected degree=2 as best.
    Metrics: Train_R2=0.668 | CV10_R2=0.417 | OOD_R2=0.052 | Overfit_suspected


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB, X165.2, X165.5"


  Target=PCEmax | Trying degree=1
  Target=PCEmax | Trying degree=2
  Target=PCEmax | Trying degree=3
    Selected degree=1 as best.
    Metrics: Train_R2=0.732 | CV10_R2=0.619 | OOD_R2=0.312 | Overfit_suspected
